# 10 — Iran-conflict event study

**Onderzoekshypothese**: sinds de Iran-oorlog op 28 februari 2026 gebruikt Donald
Trump zijn Truth Social account doelbewust om de olieprijs te beïnvloeden.

**Aanpak**: event-study op intraday WTI, S&P 500 en XLE (Energy Sector ETF) rond
Trump's Iran-gerelateerde posts. Vergelijking met (a) controle-posts in dezelfde
periode en (b) baseline volatiliteit zonder posts.

**Data sources:**
- `data/raw/posts_live.parquet` — Trump posts vanaf 2026-02-28 (via trumpstruth.org RSS)
- `data/raw/trump_truth_archive.csv` — historische posts t/m 2026-04-23 (Kaggle)
- yfinance — WTI (`CL=F`), S&P 500 (`SPY`), Energy sector ETF (`XLE`)

**Plan:**
1. Combineer live + archive posts naar één tijdreeks.
2. Filter Iran-gerelateerde posts met keyword-lijst.
3. Download intraday + daily marktdata sinds conflictstart.
4. Event-study: voor elke Iran-post, prijsbeweging in [t-1h, t+1h, t+4h, t+24h].
5. Aggregeer + vergelijk met controle-posts.
6. Anchor events: 5 zelf-gekozen ankermomenten markeren op timeline.
7. Classificeer Iran-posts met onze eigen sentiment + toxicity models (notebook 08/09).
8. Identifeer top-10 meest impactvolle posts voor scriptie-narratief.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
from datetime import date, datetime, timedelta
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yfinance as yf
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore")

CONFLICT_START = pd.Timestamp("2026-02-28", tz="UTC")
TODAY = pd.Timestamp.now(tz="UTC").normalize()
print(f"Conflict period: {CONFLICT_START.date()} → {TODAY.date()}  ({(TODAY - CONFLICT_START).days} dagen)")

## 1. Load + combineer posts

Live posts (vanaf 28 feb 2026) + Kaggle archive (tot 23 apr 2026) → één DataFrame.


In [ ]:
# Live scraped posts (run eerst `python -m src.data.scrape_trumpstruth_rss --start 2026-02-28`)
live_path = Path("../data/raw/posts_live.parquet")
if not live_path.exists():
    raise FileNotFoundError(
        "Run eerst de scraper:\n"
        "  python -m src.data.scrape_trumpstruth_rss --start 2026-02-28"
    )
live = pd.read_parquet(live_path)
live["timestamp_utc"] = pd.to_datetime(live["timestamp_utc"], utc=True)
print(f"Live posts: {len(live):,}  |  {live['timestamp_utc'].min()} → {live['timestamp_utc'].max()}")

# Optioneel: archive voor extra coverage (post_ids matchen niet noodzakelijk met live)
archive_path = Path("../data/raw/trump_truth_archive.csv")
if archive_path.exists():
    archive = pd.read_csv(archive_path, low_memory=False)
    archive["timestamp_utc"] = pd.to_datetime(archive["created_at"], utc=True, format="ISO8601")
    archive = archive[archive["timestamp_utc"] >= CONFLICT_START].dropna(subset=["text"])
    archive = archive[["id", "timestamp_utc", "text", "url"]].rename(columns={"id": "post_id"})
    archive["post_id"] = archive["post_id"].astype(str)
    print(f"Archive (vanaf 28 feb): {len(archive):,} posts")
else:
    archive = pd.DataFrame()

# Combineer + dedupe
combined = pd.concat([live, archive], ignore_index=True, sort=False)
combined["post_id"] = combined["post_id"].astype(str)
combined = combined.drop_duplicates(subset="post_id", keep="first")
combined = combined[combined["timestamp_utc"] >= CONFLICT_START].sort_values("timestamp_utc").reset_index(drop=True)
print(f"\nTotal unique posts since conflict start: {len(combined):,}")

## 2. Filter Iran-gerelateerde posts

Keyword-lijst is breed gehouden — review de output handmatig en pas aan als nodig.


In [ ]:
IRAN_KEYWORDS = [
    # Iran-specifiek
    "iran", "iranian", "tehran", "ayatollah", "khamenei", "raisi", "pezeshkian",
    "revolutionary guard", "irgc",
    # Geopolitiek
    "hormuz", "strait of hormuz", "persian gulf",
    "israel", "israeli", "netanyahu", "idf", "tel aviv",
    "houthi", "yemen", "hezbollah", "lebanon",
    # Energiebeleid
    "opec", "saudi", "saudi arabia", "riyadh", "mbs",
    "oil", "crude", "petroleum", "barrel", "gasoline",
    "drill", "drilling", "pipeline", "energy independence",
    # Militaire actie
    "strike", "airstrike", "missile", "nuclear",
    "sanctions", "embargo", "tariff",
]

# Case-insensitive substring match
pattern = "|".join(IRAN_KEYWORDS)
mask_iran = combined["text"].str.lower().str.contains(pattern, na=False, regex=True)

iran_posts = combined[mask_iran].copy()
control_posts = combined[~mask_iran].copy()

print(f"Iran-related posts:  {len(iran_posts):,}  ({len(iran_posts)/len(combined)*100:.1f}%)")
print(f"Control posts:       {len(control_posts):,}")
print(f"\nFirst 5 Iran posts:")
for _, row in iran_posts.head(5).iterrows():
    print(f"  [{row['timestamp_utc']}]  {row['text'][:130]}")

## 3. Download marktdata sinds conflictstart

Drie tickers, 2 granulariteiten:
- **Daily** (full window 28 feb → vandaag) voor de overall picture.
- **Hourly** (laatste 60 dagen via yfinance limiet) voor intraday event-study.


In [ ]:
TICKERS = {
    "WTI": "CL=F",
    "SPY": "SPY",
    "XLE": "XLE",
}

def download_market(ticker_alias: str, ticker_symbol: str, start: pd.Timestamp, interval: str = "1d") -> pd.DataFrame:
    df = yf.download(ticker_symbol, start=start.date(), interval=interval, progress=False, auto_adjust=False)
    if df.empty:
        return df
    df.columns = [c.lower() for c in df.columns.get_level_values(0)] if df.columns.nlevels > 1 else [c.lower() for c in df.columns]
    df = df.reset_index().rename(columns={df.index.name or df.columns[0]: "datetime"})
    df.columns = [c.lower() for c in df.columns]
    if "datetime" not in df.columns:
        df = df.rename(columns={"date": "datetime"})
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_convert("UTC")
    df["ticker"] = ticker_alias
    df["return"] = np.log(df["close"] / df["close"].shift(1))
    return df


# Daily
daily_frames = []
for alias, sym in TICKERS.items():
    print(f"Downloading {alias} daily…")
    daily_frames.append(download_market(alias, sym, CONFLICT_START, interval="1d"))
daily = pd.concat(daily_frames, ignore_index=True)

# Hourly (alleen voor laatste ~60 dagen)
hourly_start = max(CONFLICT_START, pd.Timestamp.now(tz="UTC") - timedelta(days=59))
hourly_frames = []
for alias, sym in TICKERS.items():
    print(f"Downloading {alias} hourly…")
    hourly_frames.append(download_market(alias, sym, hourly_start, interval="1h"))
hourly = pd.concat(hourly_frames, ignore_index=True)

print(f"\nDaily: {len(daily):,} rows ({daily['ticker'].value_counts().to_dict()})")
print(f"Hourly: {len(hourly):,} rows from {hourly_start.date()}")

## 4. Anchor events

**Vul hieronder de 5 ankerevenementen in met echte data en korte beschrijving.**
De 5 voorgesteld door Claude zijn placeholders — pas aan naar wat jij weet over
het conflict.


In [ ]:
ANCHOR_EVENTS = [
    # (datum, label, korte beschrijving)
    ("2026-02-28", "Conflict start",       "Begin van de Iran-oorlog."),
    ("2026-03-XX", "Eerste escalatie",     "VUL IN: eerste grote escalatie-moment."),
    ("2026-04-XX", "US-positie statement", "VUL IN: Trump's belangrijkste statement over US-rol."),
    ("2026-04-XX", "OPEC respons",         "VUL IN: OPEC emergency meeting of belangrijke productie-aankondiging."),
    ("2026-05-XX", "Recent moment",        "VUL IN: meest recente ankermoment voor je verdediging."),
]

anchor_df = pd.DataFrame(ANCHOR_EVENTS, columns=["date", "label", "description"])
anchor_df["date"] = pd.to_datetime(anchor_df["date"], utc=True, errors="coerce")
print(anchor_df.to_string(index=False))

## 5. Timeline: marktprijzen + post density + ankers


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True)

# Top: WTI olie prijs (daily close)
wti = daily[daily["ticker"] == "WTI"].sort_values("datetime")
axes[0].plot(wti["datetime"], wti["close"], color="#e67e22", linewidth=1.4)
axes[0].set_ylabel("WTI close ($/bbl)")
axes[0].set_title("WTI Crude Oil — sinds conflictstart")

# Middle: SPY (S&P 500) + XLE (energy)
spy = daily[daily["ticker"] == "SPY"].sort_values("datetime")
xle = daily[daily["ticker"] == "XLE"].sort_values("datetime")
ax_spy = axes[1]
ax_xle = ax_spy.twinx()
ax_spy.plot(spy["datetime"], spy["close"], color="#3498db", linewidth=1.4, label="SPY")
ax_xle.plot(xle["datetime"], xle["close"], color="#9b59b6", linewidth=1.4, label="XLE")
ax_spy.set_ylabel("SPY ($)", color="#3498db")
ax_xle.set_ylabel("XLE ($)", color="#9b59b6")
ax_spy.set_title("S&P 500 (SPY, blue) en Energy Sector ETF (XLE, purple)")

# Bottom: Post density per dag (Iran vs control)
iran_daily = iran_posts.set_index("timestamp_utc").resample("D").size()
control_daily = control_posts.set_index("timestamp_utc").resample("D").size()
axes[2].bar(control_daily.index, control_daily.values, color="lightgray", width=0.8, label="control posts")
axes[2].bar(iran_daily.index, iran_daily.values, color="#e74c3c", width=0.8, label="Iran-related posts")
axes[2].set_ylabel("# posts per dag")
axes[2].set_title("Trump posts per dag — Iran-gerelateerd vs. rest")
axes[2].legend(loc="upper right")

# Anchor lines op alle subplots
for d, label, _ in ANCHOR_EVENTS:
    d = pd.to_datetime(d, utc=True, errors="coerce")
    if pd.isna(d):
        continue
    for ax in axes:
        ax.axvline(d, color="black", linewidth=0.6, alpha=0.5, linestyle="--")
    axes[0].annotate(label, xy=(d, axes[0].get_ylim()[1]*0.97),
                     xytext=(d, axes[0].get_ylim()[1]*0.97), rotation=90,
                     fontsize=7, ha="right", va="top", color="black", alpha=0.7)

axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.setp(axes[-1].get_xticklabels(), rotation=30)
plt.tight_layout()
plt.show()

## 6. Per-post event study (intraday)

Voor elke Iran-post, kijk naar WTI/SPY/XLE prijs t.o.v. **t-1h, t+1h, t+4h, t+24h**.
Hourly data dekt alleen de laatste ~60 dagen, dus alleen recente posts.


In [ ]:
def compute_event_returns(post_time: pd.Timestamp, market_df: pd.DataFrame,
                            windows: list = [-1, 1, 4, 24]) -> dict:
    """Voor één post, return log-returns voor de gegeven windows (uren)."""
    out = {}
    # Reference: laatste price voor de post
    pre = market_df[market_df["datetime"] <= post_time]
    if pre.empty:
        return {f"ret_{w}h": np.nan for w in windows}
    p0 = pre.iloc[-1]["close"]
    for w in windows:
        target_time = post_time + pd.Timedelta(hours=w)
        if w < 0:
            # Pre-post: kijk hoe ver terug we gingen
            pre_w = market_df[market_df["datetime"] <= target_time]
            if pre_w.empty:
                out[f"ret_{w}h"] = np.nan
                continue
            p1 = pre_w.iloc[-1]["close"]
            out[f"ret_{w}h"] = float(np.log(p0 / p1))  # return van t+w naar t (=0)
        else:
            post = market_df[market_df["datetime"] >= target_time]
            if post.empty:
                out[f"ret_{w}h"] = np.nan
                continue
            p1 = post.iloc[0]["close"]
            out[f"ret_{w}h"] = float(np.log(p1 / p0))
    return out


# Alleen voor posts in het hourly data window
recent_iran = iran_posts[iran_posts["timestamp_utc"] >= hourly_start].copy()
print(f"Iran posts in hourly window: {len(recent_iran)}")

results_rows = []
for _, row in recent_iran.iterrows():
    rec = {"post_id": row["post_id"], "timestamp_utc": row["timestamp_utc"],
            "text": row["text"]}
    for alias in TICKERS:
        market_t = hourly[hourly["ticker"] == alias].sort_values("datetime")
        rets = compute_event_returns(row["timestamp_utc"], market_t)
        for k, v in rets.items():
            rec[f"{alias}_{k}"] = v
    results_rows.append(rec)

event_df = pd.DataFrame(results_rows)
print(f"\nEvent study: {len(event_df)} Iran posts with intraday returns")
print(event_df.filter(like="WTI_ret").describe().round(5))

## 7. Aggregaat-effect: Iran posts vs. control posts


In [ ]:
# Controle-groep: random sample van non-Iran posts in zelfde window
n_control = min(500, len(control_posts[control_posts["timestamp_utc"] >= hourly_start]))
control_sample = control_posts[control_posts["timestamp_utc"] >= hourly_start].sample(
    n=n_control, random_state=42
)

control_rows = []
for _, row in control_sample.iterrows():
    rec = {"post_id": row["post_id"], "timestamp_utc": row["timestamp_utc"]}
    for alias in TICKERS:
        market_t = hourly[hourly["ticker"] == alias].sort_values("datetime")
        rets = compute_event_returns(row["timestamp_utc"], market_t)
        for k, v in rets.items():
            rec[f"{alias}_{k}"] = v
    control_rows.append(rec)
control_df = pd.DataFrame(control_rows)

# Compare distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, alias in zip(axes, TICKERS):
    col = f"{alias}_ret_1h"
    iran_vals = event_df[col].dropna()
    ctrl_vals = control_df[col].dropna()

    bins = np.linspace(
        np.percentile(np.concatenate([iran_vals, ctrl_vals]), 1),
        np.percentile(np.concatenate([iran_vals, ctrl_vals]), 99),
        50,
    )
    ax.hist(ctrl_vals, bins=bins, alpha=0.5, density=True, label=f"control (n={len(ctrl_vals)})")
    ax.hist(iran_vals, bins=bins, alpha=0.5, density=True, label=f"Iran posts (n={len(iran_vals)})", color="#e74c3c")
    ax.axvline(0, color="black", linewidth=0.5)
    t_stat, p_val = stats.ttest_ind(iran_vals, ctrl_vals, equal_var=False)
    ax.set_title(f"{alias} return [t, t+1h]\nΔμ={1e4*(iran_vals.mean()-ctrl_vals.mean()):.1f}bp  p={p_val:.3f}")
    ax.set_xlabel("log return")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Numeriek: t-test per ticker × window
test_rows = []
for alias in TICKERS:
    for w in [-1, 1, 4, 24]:
        col = f"{alias}_ret_{w}h"
        iran_vals = event_df[col].dropna()
        ctrl_vals = control_df[col].dropna()
        if len(iran_vals) < 10 or len(ctrl_vals) < 10:
            continue
        t_stat, p_val = stats.ttest_ind(iran_vals, ctrl_vals, equal_var=False)
        test_rows.append({
            "ticker": alias, "window": f"{w}h",
            "n_iran": len(iran_vals), "n_ctrl": len(ctrl_vals),
            "mean_iran_bp": iran_vals.mean()*1e4,
            "mean_ctrl_bp": ctrl_vals.mean()*1e4,
            "diff_bp": (iran_vals.mean() - ctrl_vals.mean())*1e4,
            "t_stat": t_stat,
            "p_value": p_val,
        })
tests = pd.DataFrame(test_rows).round(3)
print(tests.to_string(index=False))

## 8. Top-10 meest impactvolle Iran posts

Sorteer op absolute WTI return [t, t+1h]. Dit zijn de posts waarvan het verhaal
in je scriptie waarschijnlijk komt.


In [ ]:
event_df["WTI_abs_ret_1h"] = event_df["WTI_ret_1h"].abs()
top_impact = event_df.nlargest(10, "WTI_abs_ret_1h")

for _, row in top_impact.iterrows():
    direction = "↑" if row["WTI_ret_1h"] > 0 else "↓"
    print(f"\n[{row['timestamp_utc']}]  WTI {direction}{abs(row['WTI_ret_1h'])*100:.2f}%  "
          f"SPY {row['SPY_ret_1h']*100:+.2f}%  XLE {row['XLE_ret_1h']*100:+.2f}%")
    print(f"  {row['text'][:300]}")

## 9. Sentiment + toxicity van Iran-posts

Gebruik onze eigen classifiers (notebook 08/09) om de Iran-posts te
karakteriseren. Verschilt hun sentiment/toxicity van non-Iran posts?


In [ ]:
# Load eigen classifiers
try:
    sentiment_vec = joblib.load("../models/sentiment/vectorizer.joblib")
    sentiment_clf = joblib.load("../models/sentiment/logistic_l1.joblib")
    toxicity_vec = joblib.load("../models/toxicity/vectorizer.joblib")
    toxicity_clf = joblib.load("../models/toxicity/logistic_l1.joblib")
    CLASSIFIERS_AVAILABLE = True
except FileNotFoundError as e:
    print(f"Classifiers niet gevonden — run eerst notebook 08 en 09. {e}")
    CLASSIFIERS_AVAILABLE = False

if CLASSIFIERS_AVAILABLE:
    from src.data.preprocess import clean_text

    def score_posts(df_posts):
        cleaned = df_posts["text"].apply(clean_text)
        X_sent = sentiment_vec.transform(cleaned)
        X_tox = toxicity_vec.transform(cleaned)
        df = df_posts.copy()
        df["pred_sentiment"] = sentiment_clf.predict(X_sent)
        df["proba_high_tox"] = toxicity_clf.predict_proba(X_tox)[:, 1]
        return df

    iran_scored = score_posts(iran_posts)
    control_scored = score_posts(control_posts.sample(min(2000, len(control_posts)), random_state=42))

    # Comparison
    print("\nSentiment distributie (Iran vs Control):")
    sent_compare = pd.DataFrame({
        "iran": iran_scored["pred_sentiment"].value_counts(normalize=True).round(3),
        "control": control_scored["pred_sentiment"].value_counts(normalize=True).round(3),
    })
    print(sent_compare)
    print(f"\nMean toxicity probability:")
    print(f"  Iran posts:    {iran_scored['proba_high_tox'].mean():.3f}")
    print(f"  Control posts: {control_scored['proba_high_tox'].mean():.3f}")

    t_stat, p_val = stats.ttest_ind(iran_scored["proba_high_tox"],
                                      control_scored["proba_high_tox"], equal_var=False)
    print(f"  t-stat: {t_stat:.2f}  p-value: {p_val:.4g}")

## 10. Bevindingen & scriptie-verhaal

**Vul aan met je echte resultaten:**

1. **Aggregaat-effect**: zien Iran-posts systematisch andere WTI-returns dan controle-posts?
   In welk window is het effect het sterkst (1h, 4h, 24h)? Statistisch significant?
2. **Top-10 impactful posts**: welke specifieke posts veroorzaakten de grootste WTI moves?
   Wat zeggen die posts? Is er een patroon?
3. **Sentiment/toxicity verschil**: zijn Iran-posts gemiddeld negatiever, toxischer dan
   non-Iran posts? Dat zegt iets over Trump's stijl in geopolitieke context.
4. **Anchor events**: wat gebeurde er rond de 5 ankermomenten met de marktprijzen?
   Trump-posts in 24u eromheen?

**Voor je scriptie *Results* hoofdstuk, paragraaf-structuur:**

> *"Section 5.X — Iran conflict event study: We test the hypothesis that since the
> outbreak of the Iran conflict on Feb 28, 2026, Trump has used Truth Social posts
> to influence oil prices. Using intraday WTI data and a keyword-filtered sample
> of [N] Iran-related posts, we find [statistisch significante / niet-significante]
> differences from control posts. The largest individual-post impact was observed
> on [datum] following Trump's statement about [onderwerp], which coincided with a
> [X]% movement in WTI within 1 hour. We complement this analysis with sentiment
> and toxicity classifications, finding that Iran-related posts are [meer/minder]
> negative on average than non-Iran posts."*

Dit is een **gefocust, hypothesegedreven verhaal** dat zich uitstekend leent voor
een verdediging. Eén centraal onderzoek, klaar voor je examen.
